In [0]:
# Cell 1 - SQL Database connection setup
storage_account_name = "deprojectstorage1"
storage_account_key = ""
container_processed = "processed"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# SQL connection details
sql_server = "deprojectserver2026.database.windows.net"
sql_database = "de-project-db"
sql_username = "sqladmin"
sql_password = "TableChair@1"

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_database};user={sql_username};password={sql_password};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30"

print("Connection configured!")

Connection configured!


In [0]:
# Cell 2 - Read aggregate tables from processed container
processed_base = f"abfss://{container_processed}@{storage_account_name}.dfs.core.windows.net"

agg_by_state    = spark.read.parquet(f"{processed_base}/agg_by_state")
agg_by_severity = spark.read.parquet(f"{processed_base}/agg_by_severity")
agg_by_time     = spark.read.parquet(f"{processed_base}/agg_by_time")
agg_by_weather  = spark.read.parquet(f"{processed_base}/agg_by_weather")
agg_by_hour     = spark.read.parquet(f"{processed_base}/agg_by_hour")

print("All aggregates loaded from processed container!")

All aggregates loaded from processed container!


In [0]:
# Cell 3 - Write all aggregates to Azure SQL Database via JDBC
def write_to_sql(df, table_name):
    df.write \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", table_name) \
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .mode("overwrite") \
        .save()
    print(f"✓ {table_name} loaded successfully!")

write_to_sql(agg_by_state,    "agg_by_state")
write_to_sql(agg_by_severity, "agg_by_severity")
write_to_sql(agg_by_time,     "agg_by_time")
write_to_sql(agg_by_weather,  "agg_by_weather")
write_to_sql(agg_by_hour,     "agg_by_hour")

print("\nAll tables loaded to Azure SQL Database!")

✓ agg_by_state loaded successfully!
✓ agg_by_severity loaded successfully!
✓ agg_by_time loaded successfully!
✓ agg_by_weather loaded successfully!
✓ agg_by_hour loaded successfully!

All tables loaded to Azure SQL Database!


In [0]:
# Cell 4 - Verify data loaded correctly
test = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "agg_by_state") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

print(f"agg_by_state rows in SQL: {test.count()}")
display(test.limit(5))

agg_by_state rows in SQL: 49


State,total_accidents,avg_severity
CA,1741422,2.17
FL,880159,2.14
TX,582837,2.22
SC,382557,2.11
NY,347932,2.26


In [0]:
# Cell 5 - Load agg_master to SQL DB
processed_base = f"abfss://{container_processed}@{storage_account_name}.dfs.core.windows.net"

agg_master_df = spark.read.parquet(f"{processed_base}/agg_master/")

agg_master_df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "agg_master") \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .mode("overwrite") \
    .save()

print(f"agg_master loaded: {agg_master_df.count():,} rows")

agg_master loaded: 223,059 rows
